# TF-IDF Pipeline

Firstly we will load the data files and create the TF-IDF vector, and afterwards we train a simple logistic regression classifier to later compare with the GNN approach

In [17]:
#!pip install sentence-transformers pandas numpy scikit-learn torch

In [18]:
import pandas as pd
import gzip
import json
import random
import os
import urllib.request
import torch

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [19]:
# data from https://cseweb.ucsd.edu/~jmcauley/datasets/amazon/links.html

DATASET_URLS = {
    'data/books.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Books_5.json.gz',
    'data/electronics.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Electronics_5.json.gz',
    'data/movies_tv.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Movies_and_TV_5.json.gz',
    'data/cds_vinyl.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_CDs_and_Vinyl_5.json.gz',
    'data/clothing_shoes_jewelry.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Clothing_Shoes_and_Jewelry_5.json.gz',
    'data/home_kitchen.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Home_and_Kitchen_5.json.gz',
    'data/kindle_store.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Kindle_Store_5.json.gz',
    'data/sports_outdoors.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Sports_and_Outdoors_5.json.gz',
    'data/cell_phones_accessories.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Cell_Phones_and_Accessories_5.json.gz',
    'data/health_personal_care.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Health_and_Personal_Care_5.json.gz',
    'data/toys_games.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Toys_and_Games_5.json.gz',
    'data/video_games.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Video_Games_5.json.gz',
    'data/tools_home_improvement.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Tools_and_Home_Improvement_5.json.gz',
    'data/beauty.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Beauty_5.json.gz',
    'data/apps_android.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Apps_for_Android_5.json.gz',
    'data/office_products.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Office_Products_5.json.gz',
    'data/pet_supplies.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Pet_Supplies_5.json.gz',
    'data/automotive.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Automotive_5.json.gz',
    'data/grocery_gourmet_food.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Grocery_and_Gourmet_Food_5.json.gz',
    'data/patio_lawn_garden.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Patio_Lawn_and_Garden_5.json.gz',
    'data/baby.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Baby_5.json.gz',
    'data/digital_music.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Digital_Music_5.json.gz',
    'data/musical_instruments.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Musical_Instruments_5.json.gz',
    'data/amazon_instant_video.json.gz': 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Amazon_Instant_Video_5.json.gz'
}

def ensure_data_exists(local_path):
    if os.path.exists(local_path):
        print(f"-> {local_path} already exists. Skipping download.")
        return

    if local_path not in DATASET_URLS:
        raise ValueError(f"URL for {local_path} is not defined in DATASET_URLS.")

    url = DATASET_URLS[local_path]
    dir_name = os.path.dirname(local_path)
    if dir_name and not os.path.exists(dir_name):
        os.makedirs(dir_name)

    print(f"-> Downloading {url}...")
    try:
        urllib.request.urlretrieve(url, local_path)
        print(f"-> Successfully downloaded and saved to {local_path}")
    except Exception as e:
        print(f" Error downloading {url}: {e}")
        raise

def parse_json_to_dict(path):
    with gzip.open(path, 'rb') as f:
        for line in f:
            yield json.loads(line)

def load_to_df(path, max_items=10000):

    ensure_data_exists(path)

    data = []
    for i, entry in enumerate(parse_json_to_dict(path)):
        if i >= max_items:
            break

        # here we could include more stuff from the review, but for now
        # lets use just the text
        text = entry.get('reviewText', '')

        # temporary way to get category from filename
        category = path.split('/')[-1].split('.')[0]

        data.append({
            'text': f"{text}".strip(),
            'category': category
        })
    return pd.DataFrame(data)

In [20]:
datasets = [
    'books',
    'electronics',
    'movies_tv',
    'cds_vinyl',
    'clothing_shoes_jewelry',
    'home_kitchen',
    'kindle_store',
    'sports_outdoors',
    'cell_phones_accessories',
    'health_personal_care',
    'toys_games',
    'video_games',
    'tools_home_improvement',
    'beauty',
    'apps_android',
    'office_products',
    'pet_supplies',
    'automotive',
    'grocery_gourmet_food',
    'patio_lawn_garden',
    'baby',
    'digital_music',
    'musical_instruments',
    'amazon_instant_video',
]

df = pd.DataFrame()

for dataset_name in datasets:
    dataset = load_to_df(f'data/{dataset_name}.json.gz')
    df = pd.concat([ df, dataset ], ignore_index=True)

-> Downloading https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Books_5.json.gz...
-> Successfully downloaded and saved to data/books.json.gz
-> Downloading https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Electronics_5.json.gz...
-> Successfully downloaded and saved to data/electronics.json.gz
-> Downloading https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Movies_and_TV_5.json.gz...
-> Successfully downloaded and saved to data/movies_tv.json.gz
-> Downloading https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_CDs_and_Vinyl_5.json.gz...
-> Successfully downloaded and saved to data/cds_vinyl.json.gz
-> Downloading https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Clothing_Shoes_and_Jewelry_5.json.gz...
-> Successfully downloaded and saved to data/clothing_shoes_jewelry.json.gz
-> Downloading https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Home_and

In [21]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(df['text'])
y = df['category']

words = vectorizer.get_feature_names_out()
print("Words in the vocabulary:", random.choices(words, k=10))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = LogisticRegression(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(f"TF-IDF Logistic Regression Accuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred))

Words in the vocabulary: ['square', 'midnight', 'razor', 'makeup', 'earth', 'genuine', 'ps', 'puts', 'underrated', 'dress']
TF-IDF Logistic Regression Accuracy: 0.8120625


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [22]:
mock_reviews = [
    "This guitar has a great sound and is perfect for beginners.",
    "This truck has a great suspension and is perfect for beginners.",
    "My garden has lots of birds singing beautiful songs like musical instruments.",
]

mock_X = vectorizer.transform(mock_reviews)
mock_pred = clf.predict(mock_X)
for review, pred in zip(mock_reviews, mock_pred):
    print(f"Review: {review}\nPredicted Category: {pred}\n")

Review: This guitar has a great sound and is perfect for beginners.
Predicted Category: musical_instruments

Review: This truck has a great suspension and is perfect for beginners.
Predicted Category: automotive

Review: My garden has lots of birds singing beautiful songs like musical instruments.
Predicted Category: cds_vinyl



In [23]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

print(f"-> Encoding text data on {device}... (This may take a minute)")
X_embeddings = model.encode(df['text'].tolist(), show_progress_bar=True, batch_size=32)
y = df['category']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> Encoding text data on cuda... (This may take a minute)


Batches:   0%|          | 0/7500 [00:00<?, ?it/s]

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X_embeddings, y, test_size=0.2, random_state=42)
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(f"\nSBERT + Logistic Regression Accuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred))


SBERT + Logistic Regression Accuracy: 0.8149166666666666
